<a href="https://colab.research.google.com/github/mogesTesema/Machine-Learning-Mastery-With-TensorFlow/blob/main/05_transfer_learning_in_tensorflow_part_2_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transfer Learning with tensorflow part 2: Fine-Tuning

## Creating helper functions

In [ ]:
!nvidia-smi

In [ ]:
!wget https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/refs/heads/main/extras/helper_functions.py

In [ ]:
from helper_functions import plot_loss_curves,create_tensorboard_callback,confusion_matrix,load_and_prep_image,walk_through_dir,unzip_data

## Let's get some data

In [ ]:
!wget https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_10_percent.zip

In [ ]:
unzip_data('10_food_classes_10_percent.zip')

In [ ]:
walk_through_dir("10_food_classes_10_percent")

In [ ]:
# Create training and test directory paths
train_dir = "/content/10_food_classes_10_percent/train"
test_dir = "/content/10_food_classes_10_percent/test"


In [ ]:
import tensorflow as tf

In [ ]:
IMG_SIZE = (224,224)
BATCH_SIZE = 32
train_data_10_percent = tf.keras.preprocessing.image_dataset_from_directory(directory=train_dir,
                                                                            image_size=IMG_SIZE,
                                                                            label_mode='categorical',
                                                                            batch_size=BATCH_SIZE
                                                                            )
test_data = tf.keras.preprocessing.image_dataset_from_directory(directory=test_dir,
                                                                image_size=IMG_SIZE,
                                                                batch_size=BATCH_SIZE,
                                                                label_mode="categorical")


In [ ]:
train_data_10_percent

In [ ]:
train_data_10_percent.class_names

In [ ]:
for images,labels in train_data_10_percent.take(1):
  print(images,labels)

In [ ]:
resNet_model = tf.keras.applications.ResNet50(include_top=False,weights='imagenet',input_shape=(224,224,3))


In [ ]:
resNet_weights = resNet_model.get_weights()




In [ ]:
resNet_model.compile(loss='categorical_crossentropy',
                     optimizer='adam',
                     metrics=['accuracy'])
# resNet_model.fit(train_data_10_percent,epochs=3)

## Creating the model using functional API.
what is functional API actually

In [ ]:
def getMe():
  def hello(name):
    return f"hello {name} functional API"
  return hello

result = getMe()("Keras")
print(result)

## Modle 0: Building a transfer learning model using the Keras Functional API



In [ ]:
# 1. Create the base model with tf.keras.applications.resNet50
base_model = tf.keras.applications.EfficientNetB0(include_top=False)

# 2. Freeze the base model (so the underling pre-trained patterns aren't updated durring training)
base_model.trainable = False

# 3. Create inputs into our model
inputs = tf.keras.layers.Input(shape=(224,224,3), name="input_layer")

# 4. If you using a model like ResNet50V2 you will need to normalize inputs
# x = tf.keras.layers.experimental.preprocessing.Rescaling(1/255.)(inputs)

# 5. Pass the inputs to the base_model
x = base_model(inputs)
print(f"Shape after passing inputs through base model: {x.shape}")

# 6. Average pool the outputs of the base model (aggregate all the the most important information reduce number of computations)
x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling_layer")(x)
print(f"Shape after global average 2D: {x.shape}")

# 7. Create the output activation layer
outputs = tf.keras.layers.Dense(10,activation="softmax",name="output_layer")(x)

# 8. Combine the inputs with the outputs into a model
model_0 = tf.keras.Model(inputs,outputs)

# 9. compile the model
model_0.compile(loss="categorical_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

# 10. Fit the model and save its history
model_0_history = model_0.fit(train_data_10_percent,
                              epochs=5,
                              steps_per_epoch=len(train_data_10_percent),
                              validation_data= test_data,
                              validation_steps= len(test_data),
                              callbacks=[create_tensorboard_callback("TensorFlow_Hub","Fine_tuned_EfficientNet")]
                              )




In [ ]:
plot_loss_curves(model_0_history)

In [ ]:
# Evaluate the tuned model
model_0.evaluate(test_data)

In [ ]:
for layer_number, layer in enumerate(base_model.layers):
  print(layer_number,layer.name)

In [ ]:
base_model.summary()

In [ ]:
# tf.keras.utils.plot_model(base_model)

In [ ]:
model_0.summary()

## Getting a feature vector from a trained model

In [ ]:
# # Define the input shape
# input_shape = (1,4,4,3)
# # Create a random tensor
# tf.random.set_seed(42)
# input_tensor = tf.random.normal(input_shape)
# # pass the random tensor through a global average pooling 2D layer
# global_average_pooled_tensor = tf.keras.layers.GlobalAveragePooling2D(input_tensor)
# global_average_pooled_tensor

## Running a series of transfer learning experiments
we've seen the incredible results transfer learning can get with only 10% of the training data, but how does it go with 1% of the training data.. how about we set up a bunch of experiments to find out:
1. `model_1`  - use feature extracion transfer learning with 1% of the training data with data augmentation
2. `model_2` - use feature extracctio transfer learning with 10% of the training data with data augmentation
3. `model_3` - use fine-tuning transfer learning on 10% of the training data with data augmentation
4. `model_4` - use fine-turning transfer learning on 100% of the training data with data augmentation
**Note:** throught all experiments the same test dataset will be used to evaluate our model.. this ensures consisency across evaluation metrics.


In [ ]:
!wget https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_1_percent.zip

In [ ]:
unzip_data("10_food_classes_1_percent.zip")

In [ ]:
walk_through_dir("10_food_classes_1_percent")

In [ ]:
train_dir_1_percent = "10_food_classes_1_percent/train"
test_dir = "10_food_classes_1_percent/test"


In [ ]:
# Set up data loader
IMG_SIZE = (224,224)
BATCH_SIZE = 32
train_data_1_percent = tf.keras.preprocessing.image_dataset_from_directory(directory=train_dir_1_percent,
                                                             image_size=IMG_SIZE,
                                                             label_mode="categorical",
                                                             batch_size=BATCH_SIZE,
                                                             )
test_data = tf.keras.preprocessing.image_dataset_from_directory(directory=test_dir,
                                                  image_size=IMG_SIZE,
                                                  batch_size=BATCH_SIZE,
                                                  label_mode="categorical",
                                                  )

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import preprocessing


In [ ]:
data_augmentation = tf.keras.Sequential([
    # tf.keras.layers.Rescaling(1/255.), # EfficientNet has rescaling built-in
    # tf.keras.layers.Input(shape=(224,224,3)),
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomHeight(0.2),
    tf.keras.layers.RandomWidth(0.2),
    # tf.keras.layers.RandomErasing(factor=1,scale=(0.02,0.03)),
    tf.keras.layers.RandomZoom(height_factor=(0.1,0.15)),
    ],name="data_augmentation")

In [ ]:
# view random image and compare with original image
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os
import random
target_class = random.choice(train_data_1_percent.class_names)
target_dir = "10_food_classes_1_percent/train/" + target_class
random_image = random.choice(os.listdir(target_dir))
random_image_path = os.path.join(target_dir,random_image)
img = mpimg.imread(random_image_path)
plt.imshow(img)
plt.title(f"original random image from class: {target_class}")
plt.axis(False)
plt.figure();
augmented_imag = data_augmentation(img)
print(augmented_imag.shape)
plt.imshow(augmented_imag/255.)
plt.title(f"augmented random image from class: {target_class}")
plt.axis(False)


## Model 1: feature extraction transfer learning with 1% of the data with data augmentation

In [ ]:
#data augmentation with functional layers
# Recommended way for TF ≥ 2.17
inputs = tf.keras.Input(shape=(224, 224, 3), name="input_layer")

# Apply augmentation functionally (not in a Sequential)
x = tf.keras.layers.RandomFlip("horizontal")(inputs)
x = tf.keras.layers.RandomRotation(0.2)(x)
x = tf.keras.layers.RandomHeight(0.2)(x)
x = tf.keras.layers.RandomWidth(0.2)(x)
x = tf.keras.layers.RandomZoom(0.1, 0.15)(x)
# Setup input shape and base_model
input_shape = (224,224,3)
base_model = tf.keras.applications.EfficientNetB0(include_top=False,input_shape=(224,224,3))
base_model.trainable = False

# Create the input layer
# inputs = layers.Input(shape=input_shape,name="input_layer")

# add in data augmentation Sequential model as a layer
# x = data_augmentation(inputs)

# Give base_model the inputs( after augmentation) and don't train it
x = base_model(x,training=False)

# pool output features of the base model
x = layers.GlobalAveragePooling2D(name="global_average_pooling_layer")(x)

# put a dense layer on as the output
outputs = layers.Dense(10,activation="softmax",name="output_layer")(x)

# Make a model using the inputs and outputs
model_1 = keras.Model(inputs=inputs,outputs=outputs)

# compile the model
model_1.compile(loss="categorical_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])
# fit the model
model_1_history = model_1.fit(train_data_1_percent,
                              epochs=5,
                              steps_per_epoch=len(train_data_1_percent),
                              validation_data=test_data,
                              validation_steps=(int(0.25*len(test_data))),
                              callbacks=[create_tensorboard_callback("TensorFlow_Hub","1_percent_data_exprt")]
                              )










In [ ]:
plot_loss_curves(model_1_history)

In [ ]:
model_1.summary()

In [ ]:
# Evaluate with test dataset
model_1.evaluate(test_data)

## Model 2: feature extraction transfer learning model with 10% of data and data augmentation

In [ ]:
# Get 10% of data
train_dir_10_percent = "/content/10_food_classes_10_percent/train"
test_dir = "/content/10_food_classes_10_percent/test"


In [ ]:
# Set data inputs
IMG_SIZE = (224,224)
train_data_10_percent = tf.keras.preprocessing.image_dataset_from_directory(train_dir_10_percent,
                                                                            label_mode="categorical",
                                                                            image_size=IMG_SIZE
                                                                            )
test_data = tf.keras.preprocessing.image_dataset_from_directory(test_dir,
                                                                label_mode="categorical",
                                                                image_size=IMG_SIZE)

In [ ]:
# Create model 2 with data augmentation built in
from tensorflow.keras import layers
inputs = layers.Input(shape=(224,224,3),name="input_layer")
x = layers.RandomFlip("horizontal")(inputs)
x = layers.RandomHeight(0.2)(x)
x = layers.RandomWidth(0.2)(x)
x = layers.RandomZoom(0.2)(x)
x = layers.RandomRotation(0.2)(x)

# setup the input shape to our model
input_shape = (224,224,3)
# create a frozen base model
base_model = tf.keras.applications.EfficientNetB0(include_top=False)
base_model.trainable = False
x = base_model(x,training=False)
x = layers.GlobalAveragePooling2D(name="Global_average_pooling_layer")(x)
outputs =  layers.Dense(10,activation="softmax",name="output_layer")(x)
model_2 = tf.keras.Model(inputs,outputs)

# compile the model
model_2.compile(loss="categorical_crossentropy",
                optimizer="adam",
                metrics=["accuracy"])

# Fit the model
model_2_history = model_2.fit(train_data_10_percent,
                              epochs=5,
                              steps_per_epoch=len(train_data_10_percent),
                              validation_data=test_data,
                              validation_steps=int(0.25*len(test_data)))



In [ ]:
# set checkpoint path
checkpoint_path = "ten_percent_checkpoints_weights/ckpt/checkpoint.weights.h5"
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_path,
                                                        save_weights_only=True,
                                                        save_best_only=False,
                                                        # monitors="val_accuracy",
                                                        mode="max",
                                                        save_freq="epoch",
                                                        verbose=1
                                                        )

In [ ]:
# fit the model saving checkpoints every epoch
initial_epochs = 5
model_2_history = model_2.fit(train_data_10_percent,
                              epochs=initial_epochs,
                              steps_per_epoch=len(train_data_10_percent),
                              validation_data=test_data,
                              validation_steps=int(0.25*len(test_data)),
                              callbacks=[create_tensorboard_callback("TensorFlow_Hub","10_percent_data_aug"),checkpoint_callback]
                              )


In [ ]:
plot_loss_curves(model_2_history)

In [ ]:
model_0.evaluate(test_data)

In [ ]:
results_10_percent = model_2.evaluate(test_data)

In [ ]:
# Load in saved model and evaluate model
model_2.load_weights(checkpoint_path)


In [ ]:
loaded_weights_model_result = model_2.evaluate(test_data)

In [ ]:
results_10_percent == loaded_weights_model_result

In [ ]:
results_10_percent,loaded_weights_model_result

In [ ]:
import numpy as np
np.isclose(np.array(results_10_percent),np.array(loaded_weights_model_result))

## Model 3: Fine-tuning an existing model on 10% of the data
**Note:** Fine-tuning usually works best after training a feature extraction model for a few epochs with large amounts of custom data

In [ ]:
model_2.layers

In [ ]:
for layer in model_2.layers:
  print(layer,layer.trainable)


In [ ]:
for i, layer in enumerate(model_2.layers[6].layers):
  print(i,layer.name,layer.trainable)

In [ ]:
print(len(model_2.layers[6].trainable_variables))

In [ ]:
# clone model_2
model_3 = tf.keras.models.clone_model(model_2)
#build
model_3.build(model_2.input_shape)
# used model_2 weights
model_3.set_weights(model_2.get_weights())

In [ ]:
model_3.summary(),model_2.summary()

In [ ]:
# Unfreeze the base model first
model_3.layers[6].trainable = True

# Freeze all layers except the last 10 in the base model
for layer in model_3.layers[6].layers[:-10]:
  layer.trainable = False
for layer in model_3.layers[6].layers[-10:]:
  layer.trainable = True

In [ ]:
len(model_3.layers[6].trainable_variables)

In [ ]:
for i, layer in enumerate(model_3.layers[6].layers):
  print(i,layer.name,layer.trainable)

In [ ]:
# Recompile (  we have to recompile our models every time we make a change)
model_3.compile(loss="categorical_crossentropy",
                optimizer=tf.keras.optimizers.Adam(2e-4),
                metrics=["accuracy"])

In [ ]:
# Check which layers are tunable
for layer in model_3.layers:
  print(layer,layer.trainable)
for layer_number, layer in enumerate(model_3.layers[6].layers):
  print(layer_number,layer.name,layer.trainable)

In [ ]:
# fine tune model_3 with 10 percent of training data
fine_tune_epochs = 5
total_epochs = initial_epochs + fine_tune_epochs
model_3_history = model_3.fit(train_data_10_percent,
                              epochs=total_epochs,
                              initial_epoch=model_2_history.epoch[-1],
                              steps_per_epoch=len(train_data_10_percent),
                              validation_data=test_data,
                              validation_steps=int(0.25*len(test_data)),
                              callbacks=[create_tensorboard_callback("TensorFlow_Hub","10_percent_fine_tune")]
                              )

In [ ]:
# visualize model_3 loss and accuracy curves
plot_loss_curves(model_3_history)

In [ ]:
!ls


## Model_4 fine tuning with  100% of 10 class data

In [ ]:
!wget https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_all_data.zip

In [ ]:
# load all data
unzip_data("10_food_classes_all_data.zip")




In [ ]:
walk_through_dir("10_food_classes_all_data")

In [ ]:
# load training and testing data
train_dir = "10_food_classes_all_data/train"
test_dir = "10_food_classes_all_data/test"
# Set up data loader
IMG_SIZE = (224,224)
BATCH_SIZE = 32 
train_data_all = tf.keras.preprocessing.image_dataset_from_directory(directory=train_dir,
                                                                    image_size=IMG_SIZE,
                                                                    label_mode='categorical',
                                                                    batch_size=BATCH_SIZE
                                                                    )
test_data = tf.keras.preprocessing.image_dataset_from_directory(directory=test_dir,
                                                                  image_size=IMG_SIZE,
                                                                  batch_size=BATCH_SIZE,
                                                                  label_mode="categorical")

In [ ]:
#visualize random image
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os
import random   
target_class = random.choice(train_data_all.class_names)
target_dir = "10_food_classes_all_data/train/" + target_class   

random_image = random.choice(os.listdir(target_dir))
random_image_path = os.path.join(target_dir,random_image)
img = mpimg.imread(random_image_path)
plt.imshow(img)
plt.title(f"random image from class: {target_class}")
plt.axis(False);
plt.show()



In [ ]:
# import EfficientNetB0 and preprocessing layer
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

In [ ]:
# model_4 with EfficientNetB0 base model and all data
# setup the input shape to our model
input_shape = (224,224,3)
# create a frozen base model
base_model = tf.keras.applications.EfficientNetB0(include_top=False,weights='imagenet',input_shape=(224,224,3))
base_model.trainable = False

# create the input layer
input_layer = tf.keras.layers.Input(shape=input_shape,name="input_layer")
# preprocessing layer
inputs = preprocess_input(input_layer)
# data augmentation layer
x = tf.keras.layers.RandomFlip("horizontal")(inputs)  
x = tf.keras.layers.RandomRotation(0.2)(x)
x = tf.keras.layers.RandomHeight(0.2)(x)
x = tf.keras.layers.RandomWidth(0.2)(x)
x = tf.keras.layers.RandomZoom(0.2)(x)
# pass augmented data to base model
x = base_model(x,training=False)
x = tf.keras.layers.GlobalAveragePooling2D(name="Global_average_pooling_layer")(x)
outputs =  tf.keras.layers.Dense(10,activation="softmax",name="output_layer")(x)
model_4 = tf.keras.Model(inputs,outputs)


In [ ]:
# feature Extraction model_4
model_4.compile(loss="categorical_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])



In [ ]:
# train model_4 for 5 epoch to get pattern for feature extractor layer
model_4_feature_extractor_history = model_4.fit(train_data_all,
                              epochs=5,
                              steps_per_epoch=len(train_data_all),
                              validation_data=test_data,
                              validation_steps=int(0.25*len(test_data)),
                              callbacks=[create_tensorboard_callback("TensorFlow_Hub","all_data_exprt")]
                              )

In [ ]:
# visualize model_4 feature extractor loss and accuracy curves
plot_loss_curves(model_4_feature_extractor_history)

In [ ]:
# visualize model_4 trainable and non-trainable layers
for layer in model_4.layers:
  print(layer,layer.trainable)

In [ ]:
# unfreeze top 10 layers
for layer in model_4.layers[6].layers[-10:]:
    layer.trainable = True
# visualize model_4 trainable and non-trainable layers
for layer in model_4.layers[6].layers[-15:]:
    print(layer,layer.trainable)


In [ ]:
# compile model_4 for fine-tuning
model_4.compile(loss="categorical_crossentropy",
                optimizer=tf.keras.optimizers.Adam(2e-4),
                metrics=["accuracy"])


In [ ]:
# fine tune model_4 with all data
fine_tune_epochs = 5
total_epochs = 5 + fine_tune_epochs
model_4_fine_tune_history = model_4.fit(train_data_all,
                              epochs=total_epochs,
                              initial_epoch=model_4_feature_extractor_history.epoch[-1],
                              steps_per_epoch=len(train_data_all),
                              validation_data=test_data,
                              validation_steps=int(0.25*len(test_data)),
                              callbacks=[create_tensorboard_callback("TensorFlow_Hub","all_data_fine_tune")]
                              )

In [ ]:
# visualize fine tuned model_4 loss curves
plot_loss_curves(model_4_fine_tune_history)

In [ ]:
# evaluate model_4 on test data
model_4.evaluate(test_data)

In [ ]:
# save model_4 as SavedModel format
model_4.export("saved_models/food_vision_model_4")


In [ ]:
# load the saved model
loaded_model_4 = keras.layers.TFSMLayer("saved_models/food_vision_model_4", call_endpoint='serve')

In [ ]:
x_batch, y_batch = next(iter(test_data))
preds = loaded_model_4(x_batch)


In [ ]:
from google.colab import files
import shutil

shutil.make_archive("food_vision_model_4", "zip", "saved_models/food_vision_model_4")
files.download("food_vision_model_4.zip")


In [ ]:
files.download("food_vision_model_4.zip")

In [96]:
# !python3 -m http.server 9080
!ls

10_food_classes_10_percent	helper_functions.py
10_food_classes_10_percent.zip	__MACOSX
10_food_classes_1_percent	__pycache__
10_food_classes_1_percent.zip	sample_data
10_food_classes_all_data	saved_models
10_food_classes_all_data.zip	ten_percent_checkpoints_weights
food_vision_model_4.zip		TensorFlow_Hub


In [ ]:
!curl --upload-file ./food_vision_model_4.zip https://transfer.sh/food_vision_model_4.zip

In [103]:
!ls


10_food_classes_10_percent	helper_functions.py
10_food_classes_10_percent.zip	__MACOSX
10_food_classes_1_percent	__pycache__
10_food_classes_1_percent.zip	sample_data
10_food_classes_all_data	saved_models
10_food_classes_all_data.zip	ten_percent_checkpoints_weights
food_vision_model_4.zip		TensorFlow_Hub


In [104]:
!cp /content/food_vision_model_4.zip /content/drive/MyDrive/


cp: cannot create regular file '/content/drive/MyDrive/': No such file or directory


In [ ]:
!cd /content
!python3 -m http.server 8000


Serving HTTP on 0.0.0.0 port 8000 (http://0.0.0.0:8000/) ...
